## 1. Read the text


In [8]:
import re, json
from pathlib import Path

SRC = Path("personal_status_laws_compilation.txt")
OUT = Path("chunks_article.json")

text = SRC.read_text(encoding="utf-8")
print(len(text), "chars")
print(text[:300])

77741 chars
=== صفحة 1 ===
مجموعة
قوانين الأحوال الشخصية

القانون رقم 25 لسنة 1920 م
القانون رقم 25 لسنة 1929م
القانون رقم 1 لسنة 2000م
القانون رقم 10 لسنة 2004 بإنشاء محاكم الأسرة
القانون رقم 11 لسنة 2004 صندوق التأمين علي الأسرة

=== صفحة 2 ===
قانون الأحوال الشخصية
رقم 25 لسنة 1920 المعدل بالقانون 25 لسنة 19


## 2. Page → law map
`"مادة 1"` appears in 6 different laws/decrees, so every article needs to know which law it belongs to.
Each PDF page belongs to one law (from the document map), so we look the law up by page number.



In [9]:
LAW_BY_PAGE = [
    # start, end, law_id,          law_name
    (2, 5,   "law25_1920",      "القانون 25 لسنة 1920"),
    (6, 15,  "law25_1929",      "القانون 25 لسنة 1929"),
    (19, 44, "law1_2000",       "القانون 1 لسنة 2000"),
    (45, 45, "decree1086_2000", "قرار وزير العدل 1086 لسنة 2000"),
    (46, 47, "decree1087_2000", "قرار وزير العدل 1087 لسنة 2000"),
    (48, 51, "decree1088_2000", "قرار وزير العدل 1088 لسنة 2000"),
    (52, 55, "decree1089_2000", "قرار وزير العدل 1089 لسنة 2000"),
    (56, 58, "decree1090_2000", "قرار وزير العدل 1090 لسنة 2000"),
    (61, 65, "law10_2004",      "القانون 10 لسنة 2004 (محاكم الأسرة)"),
]


def law_for_page(p):
    """Return (law_id, law_name) for PDF page p, or (None, None) if p is not mapped."""
    for start, end, law_id, law_name in LAW_BY_PAGE:
        if start <= p <= end:
            return law_id, law_name
    return None, None


for p in [1, 3, 20, 45, 57, 62, 66]:
    print(p, law_for_page(p))

1 (None, None)
3 ('law25_1920', 'القانون 25 لسنة 1920')
20 ('law1_2000', 'القانون 1 لسنة 2000')
45 ('decree1086_2000', 'قرار وزير العدل 1086 لسنة 2000')
57 ('decree1090_2000', 'قرار وزير العدل 1090 لسنة 2000')
62 ('law10_2004', 'القانون 10 لسنة 2004 (محاكم الأسرة)')
66 (None, None)


## 3. Article headings and ids


In [10]:
ARTICLE_RE = re.compile(r"^مادة\s+(\d+)\s*(.*)$")


def article_suffix(num, extra):
    """'1','' -> a001 | '76','مكرر' -> a076bis | '11','مكرر ثانيا' -> a011bis2"""
    s = f"a{int(num):03d}"
    if "مكرر" in extra:
        s += "bis"
        if "ثانيا" in extra:
            s += "2"
        if "ثالثا" in extra:
            s += "3"
    return s




## 4. Section titles


In [11]:
def is_heading(line):
    """Short line that is not a sentence -> probably a title."""
    return (0 < len(line.split()) <= 8
            and not line.endswith((".", ":", "،"))
            and not re.search(r"\d", line)
            and not line.startswith("ألغيت"))




In [12]:
chunks = []    # finished articles
cur = None     # the article we're filling right now
page = 0       # current PDF page

for raw in open(SRC, encoding="utf-8"):
    line = raw.strip()

    # (A) page marker -> remember the page
    m = re.match(r"^=== صفحة (\d+) ===$", line)
    if m:
        page = int(m.group(1))
        continue

    law_id, law_name = law_for_page(page)
    m = ARTICLE_RE.match(line)

    # (B) a new article starts
    if m and law_id:
        heading = []
        if cur:
            while len(cur["lines"]) > 1 and is_heading(cur["lines"][-1]):
                heading.insert(0, cur["lines"].pop())
            chunks.append(cur)
        num, extra = m.group(1), m.group(2).strip()
        article = (num + " " + re.sub(r"\(.*?\)", "", extra)).strip()   # drop "(مضافة)"
        cur = {"id": f"{law_id}_{article_suffix(num, extra)}",
               "law": law_id, "law_name": law_name, "article": article,
               "heading": " | ".join(heading), "page": page, "lines": []}
        continue

    # (C) a normal line -> add it to the current article
    if cur and line:
        if law_id != cur["law"] or line.startswith(("القانون رقم", "قرار وزير العدل رقم")):
            chunks.append(cur)
            cur = None
            continue
        cur["lines"].append(line)

if cur:
    chunks.append(cur)

print(len(chunks), "articles")

188 articles


Inspect one article (change the index to look at others):

In [13]:
c = chunks[100]
print(c["id"], "|", c["article"], "|", c["heading"], "| page", c["page"])
print("\n".join(c["lines"]))

law1_2000_a057 | 57 |  | page 38
يكون للنيابة العامة فى جميع الأحوال الطعن بطريق الاستئناف في الأحكام والقرارات الصادرة في
الدعاوى التي يوجب القانون او يجيز تدخلها فيها ويتبع فى الطعن الأحكام المنصوص في قانون
المرافعات المدنية والتجارية.


## 6. Contextual header + save


In [14]:
for c in chunks:
    body = "\n".join(c.pop("lines"))
    header = f"{c['law_name']} - مادة {c['article']}"
    if c["heading"]:
        header += f" - {c['heading']}"
    c["text"] = header + "\n" + body
    c["n_words"] = len(body.split())

ids = [c["id"] for c in chunks]
dups = sorted({i for i in ids if ids.count(i) > 1})
words = sorted(c["n_words"] for c in chunks)
print("duplicate ids:", dups or "none")
print("words  min:", words[0], " median:", words[len(words) // 2], " max:", words[-1])

OUT.write_text(json.dumps(chunks, ensure_ascii=False, indent=1), encoding="utf-8")
print(len(chunks), "chunks saved ->", OUT)

duplicate ids: none
words  min: 4  median: 46  max: 480
188 chunks saved -> chunks_article.json
